# Phase 4 — Convolutional Neural Networks: Teaching a Model to Actually See

The Phase 3 project ended with a complaint. Flattening a 28×28 image into a line of 784
numbers throws away everything that makes an image an image — after that step, the model has
no idea which pixels were next to each other.

Two things go wrong as a result:

1. **Neighbours mean nothing.** Pixel 100 and pixel 128 were vertically adjacent in the
   picture. In a flat list they're just two entries a long way apart.
2. **Every position is learned separately.** A 7 in the top-left corner and the same 7 shifted
   three pixels right look like completely unrelated inputs. The model has to learn "what a 7
   looks like" over again for every position it might appear in.

A **Convolutional Neural Network** fixes both.

### The idea, without the maths

Imagine looking for a specific pattern — say a short vertical stroke — in a big photo. You
wouldn't memorise what a vertical stroke looks like in the top-left corner, then separately
memorise it for the top-middle, and so on.

You'd learn what a vertical stroke looks like **once**, then sweep your eyes over the whole
image checking everywhere.

That sweep is a **convolution**. You take a small pattern-detector — a 3×3 window called a
**filter** or **kernel** — and slide it across every position in the image. At each stop it
reports how strongly the pattern matched there.

The output is a map of *"where did I see this pattern?"* — called a **feature map**.

### Why this is such a big win

**The same filter is used at every position.** Learn what an edge looks like once, and you can
spot edges anywhere in the image for free. That's called *weight sharing*, and it's why a CNN
needs far fewer parameters than a fully-connected layer for the same job.

You'll see it directly at the end of this notebook: **this CNN beats the Phase 3 MLP using
roughly half as many parameters.**

### What the network learns on its own

Nobody hand-writes these filters. They start random and are learned by the same gradient
descent as everything else. What emerges is consistent across almost every CNN ever trained:

- **Early layers** learn simple things — edges at various angles, patches of colour
- **Middle layers** combine those into corners, curves, textures
- **Late layers** combine *those* into recognisable parts — a loop, a crossbar, an eye, a wheel

The hierarchy isn't designed. It's just what turns out to be the most efficient way to
solve the problem.

### The vocabulary

| Term | What it actually means |
|---|---|
| `in_channels` | How many values per pixel coming in (1 = grey, 3 = red/green/blue) |
| `out_channels` | How many different patterns to look for — each gets its own feature map |
| `kernel_size` | How big the sliding window is. `3` means 3×3 |
| `stride` | How far the window jumps each step. `1` = every position, `2` = every other |
| `padding` | A border of zeros so the window can sit over edge pixels too |
| `nn.MaxPool2d` | Zoom out: keep the strongest response in each small area |
| feature map | The "where did I see this pattern" map that one filter produces |

## What this notebook covers
```
how conv changes shapes (padding & stride) -> pooling -> a full stack traced by hand
    -> building SimpleCNN -> training on MNIST -> a fair fight against the Phase 3 MLP
```

## Section 1 — `nn.Conv2d` and What It Does to Shapes

Everything flowing through a CNN keeps the shape **`(batch, channels, height, width)`**. Learn
to read that and CNNs stop being mysterious — most of the work is just tracking how each layer
changes those four numbers.

`in_channels` and `out_channels` are the two that confuse people, so:

- **`in_channels`** is how many values each pixel already has. A greyscale image has 1
  (brightness). A colour photo has 3 (red, green, blue).
- **`out_channels`** is **how many different patterns you want this layer to look for.**
  `out_channels=8` means "learn 8 different filters", and you get 8 feature maps out — one
  showing where each pattern was found.

So a layer going from 1 channel to 8 doesn't make the image more colourful. It replaces
"brightness at each position" with "how strongly did each of my 8 patterns match at each
position".

### `padding` — why the image shrinks without it

A 3×3 window needs a pixel on every side to sit around. The pixels on the very border don't
have that, so the window can't be centred there — and the output comes back smaller: 28×28
becomes 26×26.

Lose 2 pixels per layer and a deep network runs out of image. `padding=1` adds a one-pixel
border of zeros so the window *can* reach the edges, and the size is preserved exactly.

**The rule for keeping the size unchanged: `padding = (kernel_size - 1) // 2`.** For a 3×3
filter that's 1; for 5×5 it's 2. You'll write `kernel_size=3, padding=1` constantly.

### `stride` — how big the steps are

`stride=1` checks every position. `stride=2` skips every other one, which roughly halves the
height and width. It's a way of downsampling as you go, though pooling is the more common
choice for that.

In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)
# The shape every CNN tensor has: (batch, channels, height, width).
# Learn to read this and CNNs stop being mysterious -- most of the work is just
# tracking how each layer changes these four numbers.
image = torch.randn(1, 1, 28, 28)   # 1 image, 1 channel (grey), 28x28 pixels

# ── NO PADDING: THE IMAGE SHRINKS ────────────────────────────────────────────
# A 3x3 window needs a pixel on every side to sit around, and the border pixels
# don't have that -- so the window can't reach them and the output comes back
# smaller: 28 - (3 - 1) = 26.
# Lose 2 pixels per layer and a deep network eventually runs out of image.
conv_no_pad = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, stride=1, padding=0)
print(f'padding=0 : {image.shape} -> {conv_no_pad(image).shape}   <- shrinks by (kernel_size - 1)')

# ── PADDING=1: THE SIZE IS PRESERVED ─────────────────────────────────────────
# A one-pixel border of zeros lets the window reach the edges, so 28x28 stays
# 28x28. The rule for any odd filter: padding = (kernel_size - 1) // 2.
# For 3x3 that's 1 -- which is why you'll type 'kernel_size=3, padding=1' forever.
conv_same = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, stride=1, padding=1)
print(f'padding=1 : {image.shape} -> {conv_same(image).shape}   <- spatial size preserved')

# ── STRIDE=2: TAKE BIGGER STEPS ──────────────────────────────────────────────
# Instead of checking every position, check every other one. Roughly halves
# height and width. It's one way to downsample; pooling is the more common one.
conv_stride2 = nn.Conv2d(in_channels=1, out_channels=8, kernel_size=3, stride=2, padding=1)
print(f'stride=2  : {image.shape} -> {conv_stride2(image).shape}   <- roughly halves height & width')

# ── WHAT out_channels ACTUALLY MEANS ─────────────────────────────────────────
# out_channels=8 means 'learn 8 different pattern-detectors'. You get 8 feature
# maps out -- one per filter, each showing where that pattern was found.
#
# So going from 1 channel to 8 doesn't make the image more colourful. It swaps
# 'brightness at each position' for 'how strongly did each of my 8 patterns
# match at each position'.
print(f'\nconv_same.weight.shape: {conv_same.weight.shape}   <- (out_channels, in_channels, kh, kw)')

padding=0 : torch.Size([1, 1, 28, 28]) -> torch.Size([1, 8, 26, 26])   <- shrinks by (kernel_size - 1)
padding=1 : torch.Size([1, 1, 28, 28]) -> torch.Size([1, 8, 28, 28])   <- spatial size preserved
stride=2  : torch.Size([1, 1, 28, 28]) -> torch.Size([1, 8, 14, 14])   <- roughly halves height & width

conv_same.weight.shape: torch.Size([8, 1, 3, 3])   <- (out_channels, in_channels, kh, kw)


## Section 2 — Pooling: Zooming Out

After a convolution you have a detailed map of where every pattern matched. Often that's more
precision than you need — for recognising a digit, *"there's a strong vertical stroke
somewhere around here"* is more useful than the exact pixel.

**Max pooling** does that summarising. It slides a 2×2 window across and keeps only the biggest
value in each one. Height and width halve; the number of channels is untouched.

### Why keep the maximum?

Because these are *match scores*, and a big number means "this pattern matched strongly here".
Taking the max asks **"was there a strong match anywhere in this little area?"** — which is
usually the question you care about.

Three things come out of it:

1. **Less computation.** Quartering the pixel count makes every later layer cheaper.
2. **A wider view.** After pooling, each position summarises a bigger region of the original
   image, so later filters see more context.
3. **Small shifts stop mattering.** Nudge the digit one pixel right and the max over each 2×2
   block often doesn't change at all. The model becomes slightly shift-proof for free.

Note that pooling has **no parameters to learn** — it's a fixed rule, not something trained.

In [2]:
features = conv_same(image)          # (1, 8, 28, 28), carried over from Section 1

# Slide a 2x2 window and keep only the biggest value in each one.
# Biggest = strongest match, so this asks 'was this pattern found anywhere
# in this little area?' -- usually the question you actually care about.
#
# Note: pooling has NO parameters. It's a fixed rule, not something learned.
pool = nn.MaxPool2d(kernel_size=2)   # stride defaults to the window size
pooled = pool(features)

print(f'before pooling                : {features.shape}')
print(f'after MaxPool2d(kernel_size=2): {pooled.shape}   <- halves height & width, channels unchanged')

before pooling                : torch.Size([1, 8, 28, 28])
after MaxPool2d(kernel_size=2): torch.Size([1, 8, 14, 14])   <- halves height & width, channels unchanged


## Section 3 — Tracing a Full Stack by Hand

The standard CNN building block is three layers repeated:

```
Conv2d  ->  ReLU  ->  MaxPool2d
```

*Look for patterns → keep the strong responses → zoom out.* Stack a few of these and you have
a CNN.

Watch the trade being made as data flows through — **spatial size shrinks while channels
grow**:

```
(1, 28, 28)   1 channel,  28x28 pixels   "brightness, precisely located"
(8, 14, 14)   8 channels, 14x14          "8 simple patterns, roughly located"
(16, 7, 7)   16 channels,  7x7           "16 richer patterns, loosely located"
```

That's the deal a CNN makes at every layer: **give up "exactly where" to gain "what pattern".**
Early on you know precisely where things are but only see simple edges. Deeper in you're fuzzy
about position but recognising complex shapes — which is the right trade, because *what* the
digit is matters and *precisely where* it sat does not.

### Do this by hand once

Run the cell below and follow the shapes. That final `16 * 7 * 7 = 784` is where the number in
the next section's classifier comes from — it isn't magic, it's just what's left after two
rounds of halving.

**Getting that number wrong is the single most common CNN error.** A shape mismatch at the
first `Linear` layer means you miscounted here. When it happens — and it will — print the
shape at each step exactly like this cell does, and the mistake becomes obvious immediately.

In [3]:
# Trace the shapes by hand. Do this once properly and CNN shape errors stop
# being frightening -- when one hits, you print exactly like this to find it.
x = torch.randn(1, 1, 28, 28)
print(f'input             : {x.shape}')

# BLOCK 1: look for 8 patterns -> keep strong responses -> zoom out
x = nn.Conv2d(1, 8, kernel_size=3, padding=1)(x)   # padding=1 keeps it 28x28
x = nn.ReLU()(x)                                    # negatives to 0
x = nn.MaxPool2d(2)(x)                              # 28x28 -> 14x14
print(f'after conv1+pool1 : {x.shape}   <- 28x28 -> 14x14, 1 -> 8 channels')

# BLOCK 2: same shape again. Note the trade -- spatial size shrinks (14->7)
# while channels grow (8->16). Giving up 'exactly where' to gain 'what pattern'.
x = nn.Conv2d(8, 16, kernel_size=3, padding=1)(x)
x = nn.ReLU()(x)
x = nn.MaxPool2d(2)(x)                              # 14x14 -> 7x7
print(f'after conv2+pool2 : {x.shape}   <- 14x14 -> 7x7, 8 -> 16 channels')

# Flatten what's left, ready for an ordinary Linear layer.
# 16 channels x 7 x 7 = 784. THIS is where the number in Section 4 comes from --
# and miscounting it is the most common CNN bug there is.
x = x.view(x.size(0), -1)
print(f'flattened         : {x.shape}   <- 16 * 7 * 7 = {16 * 7 * 7}, ready for a Linear classifier head')

input             : torch.Size([1, 1, 28, 28])
after conv1+pool1 : torch.Size([1, 8, 14, 14])   <- 28x28 -> 14x14, 1 -> 8 channels
after conv2+pool2 : torch.Size([1, 16, 7, 7])   <- 14x14 -> 7x7, 8 -> 16 channels
flattened         : torch.Size([1, 784])   <- 16 * 7 * 7 = 784, ready for a Linear classifier head


## Section 4 — Building `SimpleCNN`

Almost every CNN you'll meet splits into two halves, and it's worth naming them:

- **`features`** — the conv/pool stack. Its job is to *look at the image* and turn raw pixels
  into "which patterns are present, and roughly where". Output: `(batch, 16, 7, 7)`.
- **`classifier`** — an ordinary MLP, exactly like Phase 3. Its job is to take those detected
  patterns and *decide which digit they add up to*.

This split is not a quirk of this notebook — it's the standard architecture, and it's what
makes Phase 5's transfer learning possible. If someone has already trained a `features` half on
millions of photographs, you can keep it and bolt your own small `classifier` on top.

### The flatten between them

`x.view(x.size(0), -1)` converts `(batch, 16, 7, 7)` into `(batch, 784)`, because `Linear`
layers want a flat row.

Yes — this is the same flatten that ruined things in Phase 3. The difference is **when** it
happens. Phase 3 flattened raw pixels, destroying the structure before the model could use it.
Here we flatten *after* the conv layers have already extracted spatial patterns. The structure
did its job; now we're just handing on the findings.

> ### ⚠️ A bug that produces no error
>
> It's easy to compute the classifier's output and then accidentally `return` the wrong
> variable — the flattened features instead of `self.classifier(x)`.
>
> Both are tensors, so Python is perfectly happy. You find out later, via a confusing shape
> mismatch inside the loss function, or worse, a model that trains on nonsense.
>
> **Whenever a model behaves strangely, read `forward()` line by line and check it returns
> what you think it returns.**

In [4]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()

        # PART 1 -- 'features': look at the image and work out WHICH PATTERNS
        # are present, and roughly where. Output: (batch, 16, 7, 7).
        self.features = nn.Sequential(
            nn.Conv2d(1, 8, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(8, 16, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )

        # PART 2 -- 'classifier': an ordinary MLP, exactly like Phase 3. Takes the
        # detected patterns and decides which digit they add up to.
        #
        # This two-part split is the standard CNN shape, and it's what makes
        # Phase 5 possible: keep someone else's trained 'features', swap in your
        # own 'classifier'.
        self.classifier = nn.Sequential(
            nn.Linear(16 * 7 * 7, 64), nn.ReLU(),
            nn.Linear(64, 10),   # 10 raw scores -- no Softmax, CrossEntropyLoss adds it
        )

    def forward(self, x):
        x = self.features(x)
        # Same flatten as Phase 3 -- but the TIMING is what matters. Phase 3
        # flattened raw pixels and destroyed the structure before using it.
        # Here the conv layers have already done their work; we're just passing
        # their findings on.
        x = x.view(x.size(0), -1)     # (batch, 16, 7, 7) -> (batch, 784)
        # ⚠️ Return the CLASSIFIER's output, not x. Returning the flattened
        # features here is a silent bug -- both are tensors, so Python won't
        # complain, and you find out via a confusing error inside the loss.
        return self.classifier(x)

model = SimpleCNN()
print(model)

n_params = sum(p.numel() for p in model.parameters())
print(f'\nTotal trainable parameters: {n_params:,}')

sample = torch.randn(4, 1, 28, 28)   # a batch of 4 images
print(f'\nsample batch -> logits shape: {model(sample).shape}   <- (batch, num_classes)')

SimpleCNN(
  (features): Sequential(
    (0): Conv2d(1, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(8, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Linear(in_features=784, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=10, bias=True)
  )
)

Total trainable parameters: 52,138

sample batch -> logits shape: torch.Size([4, 10])   <- (batch, num_classes)


## Section 5 — Training on MNIST

Nothing new here — the identical recipe from the Phase 3 project, with `SimpleCNN` swapped in
for the MLP.

That sameness is deliberate. **Same 5 epochs, same batch size of 64, same learning rate of
0.001, same Adam optimizer, same data.** The *only* thing that changed is the model.

That's what makes Section 6's comparison worth anything. If we'd also bumped the epochs or
tuned the learning rate, a better score would tell us nothing about whether convolution
actually helped. **When comparing two approaches, change one thing.**

You'll notice this takes longer per epoch than the MLP did, despite having fewer parameters.
Convolution does a lot of arithmetic — sliding a filter over every position is genuinely more
work than one big matrix multiply, even though it needs fewer numbers to describe.

In [5]:
import torch.optim as optim
from torchvision.datasets import MNIST
from torchvision import transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(42)

device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

transform = transforms.ToTensor()
train_data = MNIST(root="data", train=True, download=True, transform=transform)
test_data = MNIST(root="data", train=False, download=True, transform=transform)   # held-out split

train_loader = DataLoader(train_data, batch_size=64, shuffle=True)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

# Note the () -- you must BUILD the model before moving it. SimpleCNN.to(device)
# without the brackets tries to move the class itself and fails confusingly.
model = SimpleCNN().to(device)
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Identical settings to the Phase 3 MLP -- same epochs, batch size, learning rate
# and optimizer. Only the MODEL changed, which is the only way the comparison in
# Section 6 means anything. When comparing two approaches, change one thing.
epochs = 5
loss_history = []

model.train()
for epoch in range(epochs):
    total_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)   # 'outputs' -- must match the line above
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    loss_history.append(avg_loss)
    print(f"Epoch {epoch + 1}/{epochs} | Avg Loss: {avg_loss:.4f}")

plt.plot(range(1, epochs + 1), loss_history, marker="o")
plt.xlabel("Epoch")
plt.ylabel("Average training loss")
plt.title("SimpleCNN training loss on MNIST")
plt.grid(True)
plt.show()

Using device: mps


Epoch 1/5 | Avg Loss: 0.3132


Epoch 2/5 | Avg Loss: 0.0928


Epoch 3/5 | Avg Loss: 0.0667


Epoch 4/5 | Avg Loss: 0.0521


Epoch 5/5 | Avg Loss: 0.0428


## Section 6 — The Result

Same evaluation as Phase 3: `eval()` mode, `no_grad()`, accuracy on the 10,000 test images
neither model has ever seen.

And here's the payoff:

| Model | Parameters | Test accuracy |
|---|---|---|
| MLP (Phase 3) | 109,386 | 97.43% |
| **SimpleCNN** | **52,138** | **98.50%** |

**Half the parameters, and better accuracy.**

That 1.07-point gain sounds small until you convert it: the MLP got 257 test images wrong, the
CNN got 150. **It eliminated roughly 40% of the remaining errors.** Near the top of the range,
that's the honest way to read accuracy improvements — always ask what fraction of the mistakes
disappeared, not how many points were added.

And it did it with fewer than half the numbers, because of weight sharing: one filter learned
once and reused at all 784 positions, instead of separate weights for every pixel.

**This is what "the right architecture for the data" means.** Not a bigger model or a longer
training run — a model whose structure matches the structure of the problem.

In [6]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

cnn_accuracy = 100 * correct / total
print(f"SimpleCNN test accuracy: {cnn_accuracy:.2f}%  ({correct}/{total} correct)")

# The Phase 3 MLP's numbers, for a like-for-like comparison.
# Watch the parameter counts as much as the accuracy -- the CNN wins on BOTH,
# and that's the whole point of weight sharing.
mlp_accuracy = 97.43     # from phase3_project.ipynb, identical training settings
mlp_params = 109_386

print(f"\n{'Model':<12}{'Params':>12}{'Test Accuracy':>16}")
print(f"{'MLP':<12}{mlp_params:>12,}{mlp_accuracy:>15.2f}%")
print(f"{'SimpleCNN':<12}{n_params:>12,}{cnn_accuracy:>15.2f}%")

SimpleCNN test accuracy: 98.50%  (9850/10000 correct)

Model             Params   Test Accuracy
MLP              109,386          97.43%
SimpleCNN         52,138          98.50%


## Section 7 — Looking at the Mistakes

Same as the Phase 3 project: random test digits, prediction against truth, green for right and
red for wrong.

Run it a few times. The CNN's remaining errors are noticeably *harder* than the MLP's — genuinely
ambiguous scrawls, the kind where you'd want a second opinion yourself. That's a good sign:
the easy structural mistakes are gone, and what's left is real ambiguity in the handwriting.

In [7]:
model.eval()
n_show = 8
indices = torch.randint(0, len(test_data), (n_show,))

fig, axes = plt.subplots(1, n_show, figsize=(14, 2.4))
with torch.no_grad():
    for ax, idx in zip(axes, indices):
        image, true_label = test_data[idx]
        # unsqueeze(0) wraps one image into a batch of one, because the model
        # always expects a batch: (1,28,28) -> (1,1,28,28)
        input_batch = image.unsqueeze(0).to(device)
        predicted_label = model(input_batch).argmax(dim=1).item()

        ax.imshow(image.squeeze(), cmap="gray")
        is_correct = predicted_label == true_label
        ax.set_title(f"pred {predicted_label}\ntrue {true_label}",
                     color="green" if is_correct else "red")
        ax.axis("off")

plt.suptitle("SimpleCNN predictions on random test images (green = correct, red = wrong)")
plt.tight_layout()
plt.show()

## Section 8 — Saving It

Identical `state_dict` pattern to Phase 3. Save the numbers, rebuild the architecture, load
them in, and verify both models agree on the same input.

In [8]:
save_path = "simple_cnn_mnist.pth"
torch.save(model.state_dict(), save_path)
print(f"Model saved to {save_path}")

reloaded = SimpleCNN().to(device)
reloaded.load_state_dict(torch.load(save_path, map_location=device))
reloaded.eval()

with torch.no_grad():
    image, true_label = test_data[0]
    input_batch = image.unsqueeze(0).to(device)
    original_pred = model(input_batch).argmax(dim=1).item()
    reloaded_pred = reloaded(input_batch).argmax(dim=1).item()

print(f"Original model prediction : {original_pred}")
print(f"Reloaded model prediction : {reloaded_pred}   (should match)")

Model saved to simple_cnn_mnist.pth
Original model prediction : 7
Reloaded model prediction : 7   (should match)


## Phase 4 Summary

### Cheat sheet
```
Convolution -- slide a small pattern-detector over the whole image:
  nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0)
    in_channels   -> values per pixel coming in (1 grey, 3 colour)
    out_channels  -> how many patterns to look for = how many feature maps out
    padding=(kernel_size-1)//2   -> keeps height/width unchanged. For 3x3, padding=1
    stride=2                      -> skip positions, roughly halving height/width

Pooling -- zoom out, keep the strongest response:
  nn.MaxPool2d(2)   -> halves height & width, channels untouched, NO parameters to learn

The standard block:
  Conv2d -> ReLU -> MaxPool2d          (repeat; channels grow as spatial size shrinks)
  ... -> flatten -> Linear head -> logits

The standard two-part architecture:
  features   -> the conv/pool stack: WHAT patterns are here, roughly where
  classifier -> a small MLP: what do those patterns add up to
  (this split is what makes Phase 5's transfer learning possible)

Shape tracking -- do it by hand, every time:
  (1, 28, 28) -> conv+pool -> (8, 14, 14) -> conv+pool -> (16, 7, 7) -> flatten -> 784
  Get this wrong and the first Linear layer errors. Print shapes to find it.

Mistakes worth watching for:
  SimpleCNN.to(device)              -> forgot the (): needs SimpleCNN().to(device)
  loss_fn(output, labels)           -> typo; the variable above is `outputs`
  return x   in forward()           -> returned the flattened features, not the classifier's
                                        output. No error, just broken training.
  device = "cuda" if ... else "cpu" -> no mps branch, so Apple Silicon GPUs go unused

The result (5 epochs, batch 64, lr 0.001, Adam -- identical settings to the Phase 3 MLP):
  MLP        109,386 params  ->  97.43%   (257 test images wrong)
  SimpleCNN   52,138 params  ->  98.50%   (150 test images wrong)
  Half the parameters, ~40% of the remaining errors eliminated.
```

### The five things worth memorising

1. **A convolution is one pattern-detector swept across the whole image.** Learn the pattern
   once, spot it anywhere.
2. **That reuse (weight sharing) is why CNNs need fewer parameters** than fully-connected
   layers for image work.
3. **Every block trades "exactly where" for "what pattern".** Spatial size shrinks, channels
   grow.
4. **`kernel_size=3, padding=1` keeps the size unchanged.** You'll type it constantly.
5. **Track the shapes by hand.** Nearly every CNN bug is a shape you miscounted.

### Test yourself before moving on

1. An input of `(1, 3, 64, 64)` goes through `Conv2d(3, 16, kernel_size=3, padding=1)` then
   `MaxPool2d(2)`. What shape comes out? Work it out before running it.
2. Add a third conv/pool block to `SimpleCNN`. What does `16 * 7 * 7` have to become? What
   happens if you forget to update it?
3. Set every `padding=1` to `padding=0` and trace the shapes again. Where does it break?
4. Why does the CNN have fewer parameters than the MLP despite having more layers?

### Up next — Phase 5: Transfer Learning
Instead of training from scratch, take a model someone else already trained on millions of
images and adapt it to your problem. It's how nearly all real image work actually gets done.